In [ ]:
# Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved.
#
# SPDX-License-Identifier: BSD-3-Clause
#
# Portions of this file consist of AI-generated content. AI-assisted
# content has been reviewed and validated by the authors.

# Voice and MoveIt on the digital twin

This lesson is a vision-language-action loop on the twin. You speak. Gemma 4 E4B-IT,
a multimodal language model, takes that voice audio directly and turns it into one command. MoveIt plans the arm motion and the
ros2_control trajectory controller plays it. YOLO-pose and MediaPipe Hands
read your body and your hand when the command is "copy my movements".

The twin is useful because the same client drives it and the real arm:

1. **It moves like the arm** — MoveIt and the same ros2_control controllers, inside the same joint speed limits.
2. **It sees like the arm** — a camera rendered where the real one is mounted.
3. **It feels like the arm** — the gripper action reports a stall when the jaws meet something.

With those three checks done, section 4 is the console: you speak, Gemma 4 E4B-IT
picks a behavior, and MoveIt carries it out on the twin.

What is simulated: the arm, the wrist camera, and the table (MuJoCo). What is
real: your webcam and microphone, Gemma 4 E4B-IT on the iGPU, Silero VAD on the CPU,
YOLOv26s-pose on the NPU, MediaPipe Hands on the CPU, and the behaviors.
Sections 0 to 3 check the twin that `./run_notebooks.sh` already started
with `./run_pipeline.sh --digital-twin`. This kernel drives the short lesson
moves with `MoveItArm`. Section 4 sends commands to the control process.


## Compute placement

Each part of the loop runs on the processor that fits it. The lesson is the voice command and the MoveIt motion, plus pose and hands for mimic.

`./run_pipeline.sh` pins three programs to one CPU core each, from `cores` in `config/pipeline.yaml`. `./run_notebooks.sh` starts `./run_pipeline.sh --digital-twin` before this kernel, so this lesson uses that placement. The kernel only connects. `run_notebooks.sh` starts `llama-server` first, and that server keeps every core. The model stays on the iGPU.

<table style="border-collapse: collapse">
<thead>
<tr><th style="background-color: rgba(128, 128, 128, 0.2); border: 1px solid rgba(128, 128, 128, 0.4)">Task</th><th style="border: 1px solid rgba(128, 128, 128, 0.4)">Model or algorithm</th><th style="border: 1px solid rgba(128, 128, 128, 0.4)">What it does</th><th style="border: 1px solid rgba(128, 128, 128, 0.4)">Runs on</th><th style="border: 1px solid rgba(128, 128, 128, 0.4)">Runtime</th></tr>
</thead>
<tbody>
<tr style="background-color: rgba(0, 0, 0, 0)"><td rowspan="2" valign="top" style="background-color: rgba(128, 128, 128, 0.2); vertical-align: top; border: 1px solid rgba(128, 128, 128, 0.4)">Voice control</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">Silero VAD</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">marks 20 ms frames that contain speech</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">CPU (core 4)</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">ONNX Runtime</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td style="border: 1px solid rgba(128, 128, 128, 0.4)">Gemma 4 E4B-IT</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">multimodal language model; voice audio in, one command out</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">iGPU</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)"><code>llama-server</code> (HIP)</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td rowspan="2" valign="top" style="background-color: rgba(128, 128, 128, 0.2); vertical-align: top; border: 1px solid rgba(128, 128, 128, 0.4)">Your movement</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">YOLOv26s-pose</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">finds your body and wrist</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">NPU</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">ONNX Runtime, VitisAI EP</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td style="border: 1px solid rgba(128, 128, 128, 0.4)">MediaPipe Hands</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">reads the pinch and the roll</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">CPU (core 4)</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">MediaPipe Tasks</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td style="background-color: rgba(128, 128, 128, 0.2); vertical-align: top; border: 1px solid rgba(128, 128, 128, 0.4)">The object</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">YOLOv26s-detect</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">finds the named object in the wrist camera</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">NPU</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">ONNX Runtime, VitisAI EP</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td style="background-color: rgba(128, 128, 128, 0.2); vertical-align: top; border: 1px solid rgba(128, 128, 128, 0.4)">Arm motion</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">MoveIt</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">plans the command and the gripper action reports a stall</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">CPU (core 3)</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">MoveIt Task Constructor, ros2_control</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td style="background-color: rgba(128, 128, 128, 0.2); vertical-align: top; border: 1px solid rgba(128, 128, 128, 0.4)">Digital twin</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">MuJoCo</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">simulates the arm and the table</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">CPU (core 2), rendering on the iGPU</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">MuJoCo, EGL</td></tr>
</tbody>
</table>

The same three cores, by program. `llama-server` is the one this launcher starts itself, so it is not pinned.

<table style="border-collapse: collapse">
<thead>
<tr><th style="background-color: rgba(128, 128, 128, 0.2); border: 1px solid rgba(128, 128, 128, 0.4)">Program</th><th style="border: 1px solid rgba(128, 128, 128, 0.4)">What it runs</th><th style="border: 1px solid rgba(128, 128, 128, 0.4)">Runs on</th></tr>
</thead>
<tbody>
<tr style="background-color: rgba(0, 0, 0, 0)"><td style="border: 1px solid rgba(128, 128, 128, 0.4)">MuJoCo bringup</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">the twin, the wrist camera, the scene camera, and the console window</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">CPU (core 2)</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td style="border: 1px solid rgba(128, 128, 128, 0.4)">Control, <code>move_group</code>, and Servo</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">the arm client, the behaviors, planning, and the streamed joints</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">CPU (core 3)</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td style="border: 1px solid rgba(128, 128, 128, 0.4)">Perception and the mount webcam</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">YOLO pose, YOLO detect, MediaPipe, Silero, and <code>v4l2_camera</code></td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">CPU (core 4)</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td style="border: 1px solid rgba(128, 128, 128, 0.4)"><code>llama-server</code></td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">Gemma 4, started by <code>run_notebooks.sh</code> before the pipeline</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">every core; the model is on the iGPU</td></tr>
</tbody>
</table>


In [ ]:
# setup: ./run_notebooks.sh already started ./run_pipeline.sh --digital-twin.
# This kernel connects. It does not start a second twin, perception, or control.
import importlib.util
import threading, time

# hide harmless warnings from other libraries
from vvla_workshop.quiet import hush_benign_warnings

hush_benign_warnings()  # silence known-benign third-party log noise

# Keep logger.info off the cell. Jupyter prints stderr in red, and that
# stream is where an info handler lands.
import logging
logging.getLogger().setLevel(logging.WARNING)

# images (OpenCV), arrays (NumPy), plots, and the buttons and text box used in section 4
import cv2, numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from vvla_workshop.ui_poll import ui_events  # jupyter_ui_poll's, with the stop button working on cells run meanwhile

# read the pipeline settings, then switch them from the real arm to the twin
from vla_pipeline.utils.config import load_config

cfg = load_config()
cfg["robot"]["backend"] = "sim"  # the MuJoCo twin instead of the Feetech bus


## 0. A body

The twin is the MuJoCo scene under ros2_control and MoveIt. The next cell starts that bringup and connects this kernel to it with `MoveItArm`. Joint positions from here on are radians under the URDF joint names.


In [ ]:
# the MuJoCo MoveIt bringup run_notebooks.sh already started, and this kernel's arm client
from vvla_workshop.quiet import hush_benign_warnings

hush_benign_warnings()  # MoveIt INFO goes to stderr; Jupyter would show it in red
from vvla_workshop.twin_ros import TwinSession

twin = TwinSession.attach(cfg)
arm, wrist, view = twin.arm, twin.armcam, twin.view
model = arm.robot.get_robot_model()
print(f"{model.name}: {len(arm.arm_joints)} arm joints, gripper {arm.gripper_joint}")
print(f"connected to the twin on ROS domain {twin.domain_id} (./run_pipeline.sh --digital-twin)")
print(f"sim clock {arm.state_time():.3f} s")


## 1. It moves like the arm

`go_to` with a joint goal asks MoveIt for the move. MoveIt plans it and times it within each joint's `max_velocity` and `max_acceleration` in `joint_limits.yaml`, and the joint trajectory controller plays it on the twin, as it does on the arm. The samples below are the twin's joint states during a 60 degree move of the base. `go_to_rest` returns to the SRDF home pose and closes the jaws.


In [ ]:
# one planned move on the base, in radians, then back to the SRDF home pose
from sensor_msgs.msg import JointState
from vla_pipeline.robot.moveit_arm import moveit_configs

joint = "shoulder_pan_joint"
delta = 1.047  # 60 degrees
limit = moveit_configs().joint_limits["robot_description_planning"]["joint_limits"][joint]["max_velocity"]
q0 = arm.get_joints()[joint]

# record the twin's joint states, stamped on its clock, while MoveIt plans and runs the move
samples = []
sub = arm.node.create_subscription(
    JointState, "/joint_states",
    lambda m: samples.append((m.header.stamp.sec + m.header.stamp.nanosec * 1e-9, dict(zip(m.name, m.position))[joint])), 100,
)
print("move:", arm.go_to({joint: q0 + delta}))
arm.node.destroy_subscription(sub)

t, q = np.array(samples).T
t, q = t - t[0], q - q0
v = np.gradient(q, t)
print(f"peak speed {v.max():.2f} rad/s, max_velocity {limit:.2f} rad/s in joint_limits.yaml")

fig, ax = plt.subplots(figsize=(8, 3))
ax.axhline(delta, color="k", ls="--", lw=1, label="goal")
ax.plot(t, q, label="position (rad)")
ax.plot(t, v, label="speed (rad/s)")
ax.axhline(limit, color="r", ls=":", lw=1, label="max_velocity")
ax.set_xlabel("seconds on the twin's clock")
ax.legend()
plt.show()
arm.go_to_rest()


## 2. It sees like the arm

The wrist camera and the scene camera are MuJoCo camera plugins. They publish ROS images. `read()` returns `(ok, bgr)`, the same shape as a webcam. The next cell moves to the SRDF ready pose with the jaws open and shows both frames.


In [ ]:
# wrist and scene cameras from the MuJoCo plugins
arm.go_to_named("ready")
arm.go_to_named("open")
ok_w, w_img = wrist.read()
ok_v, v_img = view.read()
assert ok_w and ok_v, "a camera published no frame"
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, img, title in zip(axes, (w_img, v_img), ("wrist camera", "scene camera")):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis("off")
plt.show()


## 3. It feels like the arm

The gripper is the parallel gripper action. An empty close is asked for the SRDF close pose. The action reports whether the jaws stalled on the way there.


In [ ]:
# empty close on the gripper action, then open and park
arm.go_to_named("ready")
close = arm.named_state("close")[arm.gripper_joint]
stalled = arm.grip(close)
print(f"empty close stalled: {stalled}")
arm.go_to_named("open")
arm.go_to_rest()


## 4. Control the digital twin

From here on you drive the twin with your voice, or by typing commands. `./run_notebooks.sh` already started it, with the same command as `./run_pipeline.sh --digital-twin`. This notebook only sends and receives messages.

1. **Perception** is its own process on CPU (core 4). Gemma 4 E4B-IT (iGPU) turns a hop of voice into one command. YOLOv26s-pose (NPU) and MediaPipe Hands (CPU) watch the mount webcam. YOLOv26s-detect (NPU) reads the wrist camera.
2. **Control** is its own process on CPU (core 3). It keeps the arm client and runs the behavior for that command.
3. **The console window** is its own process on CPU (core 2). The picture in this notebook draws the same publications. It does not run a model and it does not move the arm.

Sections 0–3 use the arm client in this kernel for the short lesson moves. From here, the control process runs the behaviors.


In [ ]:
# The twin from section 0 is the one run_notebooks.sh started. Park it before voice commands.
assert twin.reset_scene(), "section 4 could not reset the twin"
assert importlib.util.find_spec("rclpy"), (
    "section 4 needs rclpy: source /opt/ros/jazzy/setup.bash, then restart ./run_notebooks.sh"
)
print(f"connected on ROS domain {twin.domain_id} (this machine only)")
print(f"arm: {type(arm).__name__}")


### Watching you

The next cell starts the mount webcam and the perception process, then shows ten seconds of the console. The tiles are the overlays perception publishes. The gripper current along the bottom is the arm's own feed, in milliamps, full scale at the servo's stall current.

In [ ]:
# Perception is already running in the pipeline. This kernel only draws what it publishes.
from rclpy.qos import DurabilityPolicy, HistoryPolicy, QoSProfile, ReliabilityPolicy
from std_msgs.msg import String

from vla_pipeline.console import TwinConsole, jpeg
from vla_pipeline.ipc import STATUS
from vla_pipeline.robot.ros2_control_arm import Spinner
from vla_pipeline.user_robot_control import PipelineContext
from vla_pipeline.wire import ConsoleLink

if "console_link" not in globals():
    ctx = PipelineContext(cfg=cfg, arm=None)
    ctx.say = lambda _message: None
    console = TwinConsole(
        ctx, view=view, hint="say 'stop robot' to halt the arm",
    )
    console.use_tiles()
    observer = Spinner("notebook_console")
    console_link = ConsoleLink(observer.node, console)

# One latched status means the models have published. This is a single wait, not a retry.
ready = threading.Event()
status_qos = QoSProfile(
    reliability=ReliabilityPolicy.RELIABLE, history=HistoryPolicy.KEEP_LAST, depth=1,
    durability=DurabilityPolicy.TRANSIENT_LOCAL,
)
observer.node.create_subscription(String, STATUS, lambda _msg: ready.set(), status_qos)
if not ready.wait(120):
    raise RuntimeError(
        "perception has not published a status. "
        "./run_notebooks.sh starts it with ./run_pipeline.sh --digital-twin; "
        "see workshop/logs/pipeline.log"
    )

preview = widgets.Image(format="jpeg", layout=widgets.Layout(width="1200px"))
display(preview)
for _ in range(10 * 15):
    preview.value = jpeg(console.compose())
    time.sleep(1 / 15)

### Listening to you

Your voice reaches the robot in three steps. The perception process from the previous cell owns the first two. This kernel runs the third.

1. **Silero VAD** (CPU) marks the 20 ms frames that contain speech and are louder than −35 dBFS. A window of that audio is sent on a hop. There is no wait for silence.
2. **Gemma 4 E4B-IT** (iGPU, through `llama-server`) is a multimodal language model. It takes the voice audio directly and answers with one robot command, for example `pick_place` with the object `ball`. Perception publishes that command.
3. The **behavior** for that command runs on the arm through MoveIt. A new command stops the one that is running.

"stop robot" stops the arm before another hop is classified as a new command. "stop" alone does nothing. Noise is ignored. `handle_transcript` (TODO 3.2) makes these checks.

Which model hears you is `voice_intent` in `config/pipeline.yaml`. Speech uses the same `gemma4-e4b-it` path as `./run_pipeline.sh`. The text box uses the keyword stand-in, the same path as `./run_pipeline.sh --text-commands`. Without the Gemma model files, that stand-in is also what hears the microphone.


In [ ]:
# The perception process already owns the microphone. A typed line in the next cell
# is published to the control process, which parses it the same way a spoken command is parsed.
print("Speak in the console cell, or type a command there.")

### Controlling the digital twin

Run the next cell, then try two things, in this order:

1. Say something that is not about the robot, for example "what's the weather like today". Gemma finds no command, the console says "I didn't catch a command in that", and the arm does not move.
2. Say "wave at me". Gemma picks `wave`, and the arm waves.

Under the console, **stop** ends the cell and parks the arm, **reset** turns gray and says "resetting..." until the arm is parked and the objects are back (another click does nothing until then), and **end session** also ends the cell (run the cell again to resume). Saying "stop" only halts the behavior, so a later command in the same session still works. The notebook runs one cell at a time, so click **stop** or **end session** (or the kernel's stop button) before you run the clean-up cell.

| Say | What happens |
|---|---|
| "copy my movements" | The arm follows your right hand: where it is in the image, how close it is to the camera, its tilt, and your pinch for the gripper. |
| "pick up the ball" (or bottle, book, toothbrush) | MoveIt plans a pick of the object the command names, lifts it, and holds it. |
| "hold this" | The jaws close on what is between them and keep holding. If you then say "copy my movements", your pinch controls the gripper. |
| "release", "drop it" | The jaws open. |
| "place it", "set it down" | After a pickup, the arm lowers the object onto the table under the jaws, opens them, and returns home. |
| "wave", "dance" | The arm waves, or dances to a beat. |
| "go home" | The arm returns to its rest pose. |
| "stop robot" | Everything stops at once. |

All commands: [Voice commands → behaviors](../../../README.md#voice-commands--behaviors).


In [ ]:
# the console image with its buttons and text box, redrawn FPS times a second
FPS = 15
# Start with a frame, not empty: JupyterLab may not re-measure an image that grows later.
screen = widgets.Image(value=jpeg(console.compose()), format="jpeg", layout=widgets.Layout(width="1200px"))
stop = widgets.Button(description="stop", button_style="danger")
reset = widgets.Button(description="reset", button_style="warning")
typed = widgets.Text(placeholder="or type a command, then Enter", continuous_update=False,
                     layout=widgets.Layout(width="400px"))
end = widgets.Button(description="end session")
finished = threading.Event()  # set by stop or end session


def submit(change):
    """Enter in the text box sends the typed command to the control process."""
    text = change["new"].strip()
    if text:
        typed.value = ""
        console_link.publish_text(text)


def end_session(_=None):
    """Ends the loop below and asks the control process to stop the behavior."""
    finished.set()
    console_link.publish_stop()


def on_reset(_):
    """Gray the button, redraw, then ask the control process to reset."""
    if reset.disabled:
        return
    reset.disabled = True
    reset.description = "resetting..."
    reset.button_style = ""
    screen.value = jpeg(console.compose())
    console_link.publish_reset()


stop.on_click(end_session)  # the red button ends the cell; saying "stop" only halts the behavior
reset.on_click(on_reset)
typed.observe(submit, names="value")
end.on_click(end_session)
display(widgets.VBox([screen, widgets.HBox([stop, reset, typed, end])]))

print("Ready. Try: 'copy my movements', 'hold this', 'pick up the ball', 'pick up the bottle', "
      "'pick up the toothbrush', 'wave', 'dance', 'go home' - or 'stop robot'.")
shown = ""
try:
    # running this cell again while it runs (JupyterLab's run and advance) also ends the session
    with ui_events(on_rerun=end_session) as poll:
        while not finished.is_set():
            t0 = time.perf_counter()
            poll(10)  # deliver button clicks and typed commands while this cell runs
            said = console.said
            if said and said != shown:
                shown = said
                print(said)
            # Stay gray until the redraw after the reset, so a click queued during it is ignored.
            if reset.disabled and not console.resetting:
                reset.description = "reset"
                reset.button_style = "warning"
                reset.disabled = False
            screen.value = jpeg(console.compose())
            time.sleep(max(0.0, 1.0 / FPS - (time.perf_counter() - t0)))
finally:
    end_session()
    print("session ended")

### Running it outside the notebook

This section ran the pipeline itself (`vla_pipeline/`), which holds the finished code of every TODO. Your versions are in `workshop/project/`:

- `models/`: `npu.py` (TODOs 1.1-1.3), `hands.py` (1.4), `llm.py` (1.5-1.6)
- `ros/`: `camera_client.py` (2.3), `intent_node.py` (2.10-2.11)
- `integration/`: `dispatch.py` (3.1-3.2), `mimic.py` (3.3), `behaviors.py` (3.4)

`python -m vvla_workshop.selftest` checks all of them; `--reference` runs the same checks on the pipeline's code. `./run_notebooks.sh` already started this section with `./run_pipeline.sh --digital-twin`. That is the same command you run in a terminal when you want the console in its own desktop window.


## Clean up

Click **stop** or **end session** under the console (or the kernel's stop button) before you run this cell. It stops perception and the mount camera, then closes the console and the twin. The microphone closes with perception.


In [ ]:
# the console cell ended its session
finished.set()
observer.close()
console.close()
twin.close()  # the arm client only; run_notebooks.sh owns the pipeline
print("twin closed")